In [42]:
from dataclasses import Field

from pydantic import BaseModel, PositiveInt, EmailStr


class Author(BaseModel):
    name: str
    age: PositiveInt
    email: EmailStr | None = None

class Book(BaseModel):
    title: str
    pages: PositiveInt
    authors: list[Author]

hobbit = Book.model_validate({
    "title": "The Hobbit",
    "pages": 300,
    "authors": [{"name": "J.R.R.", "age": 32, "email": None}]
})


print(hobbit)
# Generate a dictionary representation of the model
# {instance}.model_dump() -> dict
print(hobbit.model_dump())


title='The Hobbit' pages=300 authors=[Author(name='J.R.R.', age=32, email=None)]
{'title': 'The Hobbit', 'pages': 300, 'authors': [{'name': 'J.R.R.', 'age': 32, 'email': None}]}


In [11]:
%xmode
hobbit = Book.model_validate({
    "title": "The Hobbit",
    "pages": 300,
    "authors": [{"name": "J.R.R.", "age": -32, "email": None}]
})

Exception reporting mode: Verbose


ValidationError: 1 validation error for Book
authors.0.age
  Input should be greater than 0 [type=greater_than, input_value=-32, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

In [39]:
from pydantic_core import ValidationError

# greater_than -> gt
class GreaterThan(BaseModel):
    maiores: int = Field(gt=18)

try:
 GreaterThan(maiores=11)
except ValidationError as exc:
    print("gt ->",repr(exc.errors()[0]['msg']))

try:
    print("18 is not greater than 18")
    GreaterThan(maiores=18)
except ValidationError as exc:
    print("gt ->",repr(exc.errors()[0]['msg']))



gt -> 'Input should be greater than 18'
18 is not greater than 18
gt -> 'Input should be greater than 18'


In [40]:
# greater_than_equal -> ge
class GreaterThanEqual(BaseModel):
    maiores: int = Field(ge=18)

try:
    print("17 is not greater than or equal to 18")
    GreaterThanEqual(maiores = 17)
except ValidationError as exc:
    print("ge ->",repr(exc.errors()[0]['msg']))

17 is not greater than or equal to 18
ge -> 'Input should be greater than or equal to 18'


In [45]:
# less_than -> lt
from pydantic import BaseModel, Field, ValidationError

class Model(BaseModel):
    x: int = Field(lt=10)
try:
    Model(x=10)
except ValidationError as exc:
    print("ls:",repr(exc.errors()[0]['msg']))
    #> 'less_than'


# less_than_equal -> le
class Model(BaseModel):
    x: int = Field(le=10)
try:
    Model(x=11)
except ValidationError as exc:
    print("le:",repr(exc.errors()[0]['msg']))
    #> 'less_than_equal'

ls: 'Input should be less than 10'
le: 'Input should be less than or equal to 10'


In [47]:
from pydantic import BaseModel, conlist, ValidationError

class Post(BaseModel):
    title: str
    tags: conlist(str, min_length=1, max_length=3)

# OK
p = Post(title="Pydantic", tags=["python", "validation"])
print(p)

# Erro: lista vazia (min_length=1)
try:
    Post(title="Oops", tags=[])
except ValidationError as exc:
    print("lista vazia:",repr(exc.errors()[0]['msg']))

# Erro: lista grande demais (max_length=3)
try:
    Post(title="Oops", tags=["a", "b", "c", "d"])
except ValidationError as exc:
    print("grande demais:",repr(exc.errors()[0]['msg']))

title='Pydantic' tags=['python', 'validation']
lista vazia: 'List should have at least 1 item after validation, not 0'
grande demais: 'List should have at most 3 items after validation, not 4'


In [49]:
from decimal import Decimal
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError

# greater_than -> gt
# greater_than_equal -> ge
# less_than -> lt
# less_than_equal -> le
# multiple_of -> multiple_of
# max_digits -> max_digits
# decimal_places -> decimal_places

class DecimalRules(BaseModel):
    valor: Annotated[
        Decimal,
        Field(
            gt=Decimal("10.00"),
            ge=Decimal("10.50"),
            lt=Decimal("20.00"),
            le=Decimal("19.99"),
            multiple_of=Decimal("0.25"),
            max_digits=6,
            decimal_places=2,
            allow_inf_nan=False,
            strict=True,
        ),
    ]

# OK
ok = DecimalRules(valor=Decimal("10.50"))
print("OK ->", ok)

# Erro ge (e também gt)
try:
    print("10.00 is not greater than or equal to 10.50")
    DecimalRules(valor=Decimal("10.00"))
except ValidationError as exc:
    print("ge ->", repr(exc.errors()[0]["msg"]))

# Erro lt/le
try:
    print("20.00 is not less than 20.00")
    DecimalRules(valor=Decimal("20.00"))
except ValidationError as exc:
    print("lt/le ->", repr(exc.errors()[0]["msg"]))

# Erro multiple_of
try:
    print("10.60 is not a valid multiple of 0.25")
    DecimalRules(valor=Decimal("10.60"))
except ValidationError as exc:
    print("multiple_of ->", repr(exc.errors()[0]["msg"]))

# Erro decimal_places
try:
    print("10.555 has more than 2 decimal places")
    DecimalRules(valor=Decimal("10.555"))
except ValidationError as exc:
    print("decimal_places ->", repr(exc.errors()[0]["msg"]))

# Erro max_digits
try:
    print("12345.67 exceeds max_digits=6")
    DecimalRules(valor=Decimal("12345.67"))
except ValidationError as exc:
    print("max_digits ->", repr(exc.errors()[0]["msg"]))

OK -> valor=Decimal('10.50')
10.00 is not greater than or equal to 10.50
ge -> 'Input should be greater than or equal to 10.50'
20.00 is not less than 20.00
lt/le -> 'Input should be less than or equal to 19.99'
10.60 is not a valid multiple of 0.25
multiple_of -> 'Input should be a multiple of 0.25'
10.555 has more than 2 decimal places
decimal_places -> 'Decimal input should have no more than 2 decimal places'
12345.67 exceeds max_digits=6
max_digits -> 'Decimal input should have no more than 6 digits in total'
